# 출력파서
- 출력 파서(Output Parser)는 LLM의 출력값을 구조화된 형식으로 변환하고 답변에서 우리가 원하는 정보만 뽑아낼 때 유용하게 사용되는 도구
- 출력 파서를 사용하면 LLM의 응답을 구조화된 데이터로 받거나 원하는 정보를 손쉽게 추출하는 것이 가능
- 반면 출력 파서를 사용하지 않으면 자유 형식의 텍스트 출력을 수동으로 해석하고 필요한 정보를 추출해야 하므로 자동화가 어려움

In [1]:
from dotenv import load_dotenv
from langchain_teddynote import logging
import os

load_dotenv()

print("OpenAI 키 : ", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키 : ", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

logging.langsmith("test0914")

OpenAI 키 :  sk-proj-...
LANGSMITH 키 :  lsv2_pt_...
LangSmith 프로젝트 :  test0914
LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [ ]:
# pip install langchain langchain-openai langchain-chroma langchain-teddynote python-dotenv jupyter ipykernel

In [2]:
import os

from dotenv import load_dotenv

# 모델
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 프롬프트
from langchain_core.prompts import (
    PromptTemplate,
    ChatPromptTemplate,
    MessagesPlaceholder,
    FewShotPromptTemplate,
    FewShotChatMessagePromptTemplate,
)

# 예시 선택기 / 벡터스토어
from langchain_core.example_selectors import (
    MaxMarginalRelevanceExampleSelector,
    SemanticSimilarityExampleSelector,
)
from langchain_chroma import Chroma

# 출력 파서
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

# teddynote 유틸
from langchain_teddynote import logging
from langchain_teddynote.messages import stream_response

from pydantic import BaseModel, Field


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

load_dotenv()

print("OpenAI 키 : ", os.getenv("OPENAI_API_KEY")[:8] + "...")
print("LANGSMITH 키 : ", os.getenv("LANGSMITH_API_KEY")[:8] + "...")
print("LangSmith 프로젝트 : ", os.getenv("LANGSMITH_PROJECT"))

logging.langsmith("test0914")

OpenAI 키 :  sk-proj-...
LANGSMITH 키 :  lsv2_pt_...
LangSmith 프로젝트 :  test0914
LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [3]:
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

### 출력 파서를 쓰지 않는 경우
- 문자열 형식으로 나옴
- 특정한 항목을 꺼내오기 까다로움

In [4]:
from itertools import chain

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})
output = stream_response(answer, return_output=True)

중요한 내용 요약:

- 발신자: 김철수 (바이크코퍼레이션 상무)
- 수신자: 이은채 (Teddy International)
- 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안
- 요청 사항: ZENESIS 모델에 대한 상세 브로슈어 (기술 사양, 배터리 성능, 디자인 정보)
- 미팅 제안: 다음 주 화요일(1월 15일) 오전 10시, 귀사 사무실에서 만남 제안
- 목적: 협력 가능성 논의 및 유통 전략, 마케팅 계획 구체화

In [5]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 테스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

In [6]:
parser = PydanticOutputParser(pydantic_object=EmailSummary)

In [7]:
print(parser.get_format_instructions())

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 테스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


In [17]:
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following question in KOREAN.

QUESTION:{question}

EMAIL CONVERSATION: {email_conversation}

FORMAT: {format}
"""
)

In [18]:
prompt = prompt.partial(format = parser.get_format_instructions())

prompt

PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 테스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required

In [19]:
chain = prompt | llm

response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요."
    }
)

output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "김철수 상무가 이은채 대리에게 'ZENESIS' 자전거에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 함.",
  "date": "2024-01-15T10:00:00"
}
```

In [11]:
structured_output = parser.parse(output)
print(structured_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary="김철수 상무가 이은채 대리님에게 'ZENESIS' 자전거에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 함." date='2024-01-15T10:00:00'


In [12]:
structured_output.person

'김철수'

In [13]:
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요"
    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary="김철수 상무가 이은채 대리에게 'ZENESIS' 자전거에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 함.", date='1월 15일 오전 10시')

### with_structured_output() 바인딩
- LLM에 with_structured_output() 함수를 사용해서 출력 파서를 추가하면 출력을 Pydantic 객체로 변환할 수 있음

In [14]:
llm.invoke("대한민국의 수도는 뭐야 ?") # 이 답변은 구조화된 답변이 아님

AIMessage(content='대한민국의 수도는 서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 15, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_3ee27fbcbf', 'id': 'chatcmpl-EOYkUzUcANPTbUIVfepoSC5M9U1o1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0a7d5-1408-7641-bdcc-2cf0a4b669ff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 15, 'output_tokens': 8, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [20]:
# 우리가 원하는 건 llm_with_structured 객체에 이메일 본문(email_conversation) 내용을
# 전달해서 EmailSummary에 정의된 대로 구조화된 답변을 받는 것

llm_with_structured = ChatOpenAI(model="gpt-4o-mini", temperature=0).with_structured_output(EmailSummary)

In [21]:
anwer = llm_with_structured.invoke(email_conversation)

anwer.person

'이은채'